In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys,os
sys.path.append('..')
import style
style.load_preset(scale=1)
from pipic import consts

wavelength = 1e-4 #units? cm?
boundarySize = 4*wavelength
nx, XMin, XMax = 2**8, -16*wavelength, 16*wavelength
ny, YMin, YMax = 2**8 + int(2**8/(32*wavelength)*2*boundarySize), -16*wavelength-boundarySize, 16*wavelength+boundarySize
dx, dy = (XMax - XMin)/nx, (YMax - YMin)/ny
ts = 1/4 
timeStep = ts*dx/consts.light_velocity


amplitude_a0 = 10.
omega = 2*np.pi*consts.light_velocity/wavelength
a0 = consts.electron_mass*consts.light_velocity*omega/(-consts.electron_charge)
fieldAmplitude = amplitude_a0*a0
PulseDuration = 10e-15 

rot = -np.pi/4

In [ ]:
fig, ax = plt.subplots()


fp = './data/'
ts = 0.25
falls = {
0.001 : np.loadtxt(fp + f'/data_ts_{ts}_fall_0.001_bs_4.0.txt'),
0.01 : np.loadtxt(fp + f'/data_ts_{ts}_fall_0.01_bs_4.0.txt'),
0.1 : np.loadtxt(fp + f'/data_ts_{ts}_fall_0.1_bs_4.0.txt'),
1.0 : np.loadtxt(fp + f'/data_ts_{ts}_fall_1.0_bs_4.0.txt'),
10.0 : np.loadtxt(fp + f'/data_ts_{ts}_fall_10.0_bs_4.0.txt'),
100.0 : np.loadtxt(fp + f'/data_ts_{ts}_fall_100.0_bs_4.0.txt'),
1000.0 : np.loadtxt(fp + f'/data_ts_{ts}_fall_1000.0_bs_4.0.txt'),
}

for f in falls:
    ax.plot(falls[f][:,0],np.log(falls[f][:,1]),'-',label='fall='+str(f))




In [ ]:
fp = 'data/'

bss = {
0.5 : np.loadtxt(fp+'/data_ts_0.25_fall_1.0_bs_0.5.txt'),
1.0 : np.loadtxt(fp+'/data_ts_0.25_fall_1.0_bs_1.0.txt'),
2.0 : np.loadtxt(fp+'/data_ts_0.25_fall_1.0_bs_2.0.txt'),
4.0 : np.loadtxt(fp+'/data_ts_0.25_fall_1.0_bs_4.0.txt'),
8.0 : np.loadtxt(fp+'/data_ts_0.25_fall_1.0_bs_8.0.txt'),
#16.0 : np.loadtxt(fp+'/data_ts_0.5_fall_0.01_bs_16.0.txt')
}
for b in bss:
    plt.plot(bss[b][:,0],np.log(bss[b][:,1]),label='bs='+str(b))

#plt.vlines(2*16*np.sqrt(2)*wl/c,19,25,'k')
'''

for f in falls:
    plt.plot(falls[f][:,0],np.log(falls[f][:,1]),'--',label='dt='+f)
'''

plt.legend(frameon=False)

In [ ]:

absorption = []
fs = []
for f in falls:
    #tb = 2*12*np.sqrt(2)*wl/c
    ind = -1 #np.argwhere(falls[f][:,0]>=tb)[0]
    absorption.append(1-falls[f][ind,1]/falls[f][0,1])
    fe = float(f)#*1e18*dt
    fs.append(fe)
print(fs)
fig,ax = plt.subplots(1,3,figsize = (10,3))
ax[1].plot(np.log10(fs),absorption,'.',color='tab:blue',ms=10)
ax[1].plot(np.log10(fs),absorption,'tab:blue')
ax[1].plot(np.log10(fs[2:5]),absorption[2:5],'.',color='tab:red',ms=10)

ax[1].set_xlabel(r'Shape parameter ($\mathrm{log}_{10}(\alpha$))')
ax[0].set_ylabel(r'Attenuation ($1-u_r/u_0$)')
ax[1].set_title('Shape parameter')


def mask(x,fall,dt):
    return np.exp(fall*(np.cos(x*pi/2) - 1/np.cos(x*pi/2)))#*dt*1e16)


x = np.linspace(0,1)
for i,f in enumerate(falls):
    print(float(f))
    color = 'tab:blue'
    if i in [2,3,4]:
        color = 'tab:red'
    ax[2].plot(x,1-mask(x,float(f),dt),color=color)
    print(f)

ax[2].set_xlabel('Boundary depth')
ax[2].set_title('Attenuation function')
#ax[1].set_ylabel('Attenuation factor')
ax[2].annotate(r'Increasing $\alpha$',xy=(0,1),xytext=(1.7/4,0.33),)
ax[2].annotate('',xy=(0,1),xytext=(1*0.6,0.4),arrowprops=dict(facecolor='black',
                                                                                width=0.1,
                                                                                headwidth=10.,
                                                                                headlength=10.))

absorption = []
fs = []
for f in bss:
    #tb = 2**5*np.sqrt(2)*wl/c
    ind = -1 #np.argwhere(bs[f][:,0]>=tb)[0]
    absorption.append(1-bss[f][ind,1]/bss[f][0,1])
    fs.append(float(f))
    
ax[0].plot(fs,absorption,'.',color='tab:blue',ms=10)
ax[0].plot(fs,absorption,'tab:blue')
ax[0].set_xlabel(r'$\lambda$')
#ax[2].set_ylabel(r'Attenuation ($1-I/I_0$)')
ax[0].set_title('Boundary thickness')
fig.tight_layout()
fig.savefig('boundary_absorption.png')




In [ ]:
import glob 

files = glob.glob('./data/data_ts_*.txt')

ts0125 = []
ts00625 = []
ts05 = []

for file in files:
    d = np.loadtxt(file)
    absorption = 1-d[-1,1]/d[0,1]

    if 'ts_0.125' in file:
        fall = file.split('fall_')[1].split('_bs_')[0]
        bs = file.split('bs_')[1].split('.txt')[0]
        ts0125.append([float(fall),float(bs),absorption])
    if 'ts_0.0625' in file:
        d = np.loadtxt(file)
        fall = file.split('fall_')[1].split('_bs_')[0]
        bs = file.split('bs_')[1].split('.txt')[0]
        ts00625.append([float(fall),float(bs),absorption])
    if 'ts_0.25' in file:
        d = np.loadtxt(file)
        fall = file.split('fall_')[1].split('_bs_')[0]
        bs = file.split('bs_')[1].split('.txt')[0]
        ts05.append([float(fall),float(bs),absorption])

ts0125 = np.array(ts0125)
ts00625 = np.array(ts00625)
ts05 = np.array(ts05)


In [ ]:
from scipy.interpolate import RegularGridInterpolator


ts05_s = ts05[np.argsort(ts05[:,0])]
ind = 6
c = 0
while (c+1)*6 < ts05.shape[0] + 1:
    t = ts05_s[c*6:(c+1)*ind][:,1]
    ind_sort = t.argsort()
    ts05_s[c*6:(c+1)*ind] = ts05_s[c*6:(c+1)*ind][ind_sort]
    c += 1

x = np.log10(ts05_s[::6,0])
y = (ts05_s[:6,1])
z = ts05_s[:,2].reshape((x.shape[0],y.shape[0]))

interp = RegularGridInterpolator((x,y),z)

ix = np.linspace(x[0],x[-1],100)
iy = np.linspace(y[0],y[-1],100)
X, Y = np.meshgrid(ix,iy,indexing='ij')
Z = interp((ix[:,None],iy[None,:]))
fig,ax = plt.subplots(1,1)
c = ax.pcolormesh(X,Y,Z,vmin=0.9,vmax=1,cmap='coolwarm')
ax.scatter(np.log10(ts05[:,0]),ts05[:,1],marker='.',color='k',label='dt=0.5')

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap_cw = plt.get_cmap('coolwarm')

color_min    = cmap_cw(0)
color_center = "white"
color_max    = cmap_cw(256)
mymap = LinearSegmentedColormap.from_list(
    "cmap_name",
    [color_min, color_center, color_max]
)

x = np.meshgrid(np.linspace(0,1),np.linspace(0,1))

plt.imshow(x[0],cmap=mymap)

In [ ]:
cmap_cw = plt.get_cmap('Oranges')

color_min    = cmap_cw(0,alpha=0)
color_center = cmap_cw(128)
color_max    = cmap_cw(256)
mymap_red = LinearSegmentedColormap.from_list(
    "cmap_name",
    [color_min, color_center, color_max]
)

x = np.meshgrid(np.linspace(0,1),np.linspace(0,1))

plt.imshow(x[0],cmap=mymap_red)

In [ ]:

absorption = []
fs = []
for f in falls:
    ind = -1
    absorption.append(1-falls[f][ind,1]/falls[f][0,1])
    fe = float(f)
    print(float(f))
    fs.append(fe)

cmap_cw = plt.get_cmap('coolwarm')
colors = [cmap_cw(0),cmap_cw(256)]

fig_w = style.figsize['inch']['double_column_width']*0.9

fig,ax = plt.subplots(2,2,figsize = (fig_w,fig_w*0.8))

ax = ax.flatten()
ax[2].plot(fs,absorption,'.',color=colors[0])
ax[2].plot(fs,absorption,color=colors[0])
ax[2].plot(fs[2:5],absorption[2:5],'.',color=colors[1])

ax[2].set_xlabel(r'Shape parameter $\alpha$ ($c/\lambda$)')
ax[2].set_ylabel(r'Absorption')
ax[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
ax[2].set_xscale('log')

def mask(x,fall,dt):
    fall *= consts.light_velocity/wavelength
    return np.exp(fall*(np.cos(x*np.pi/2) - 1/np.cos(x*np.pi/2))*dt)


x = np.linspace(0,1)
for i,f in enumerate(falls):
    color = colors[0]
    if i in [2,3,4]:
        color = colors[1]
    ax[1].plot(x,1-mask(x,float(f),timeStep),color=color)

ax[1].set_xlabel('$x_r$')
ax[1].annotate(r'Increasing $\alpha$',xy=(0,0),xytext=(0.6,0.22),)
ax[1].annotate('',xy=(0.2,0.8),xytext=(0.7,0.3),arrowprops=dict(facecolor='black',
                                                            width=0.05,
                                                            headwidth=3.,
                                                            headlength=3.,
                                                            lw=0.5))

absorption = []
fs = []
for f in bss:
    ind = -1 
    absorption.append(1-bss[f][ind,1]/bss[f][0,1])
    fs.append(float(f))
    
ax[3].plot(fs,absorption,'.',color=colors[0])
ax[3].plot(fs,absorption,color=colors[0])
ax[3].set_xlabel(r'Boundary size ($\lambda$)')

ax[3].set_yticks([0.96,0.98,1])
ax[3].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))



In [ ]:

def field(r):
    r_rot = (r[0]*np.cos(rot)+r[1]*np.sin(rot), -r[0]*np.sin(rot)+r[1]*np.cos(rot))
        
    # P-polarized
    Ep_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*consts.light_velocity)**2)*np.cos(k*r_rot[0])
    Bp_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*consts.light_velocity)**2)*np.cos(k*r_rot[0])

    # S-polarized
    Es_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*consts.light_velocity)**2)*np.cos(k*r_rot[0])
    Bs_rot = -fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*consts.light_velocity)**2)*np.cos(k*r_rot[0])

    E = np.zeros((3,r.shape[-1]))
    B = np.zeros((3,r.shape[-1]))
    E[0] = Ep_rot*np.sin(-rot)
    E[1] = Ep_rot*np.cos(rot) 
    B[2] = Bp_rot

    B[0] = Bs_rot*np.sin(-rot)
    B[1] = Bs_rot*np.cos(rot) 
    E[2] = Es_rot
    return E,B


r = np.meshgrid(np.linspace(XMin/2, XMax*2, nx), np.linspace(YMin - boundarySize, YMax/2, ny), indexing='ij')
r = np.array(r).reshape((2,nx*ny))
E_field,B_field = field(r)
E_field = E_field.reshape((3,nx,ny))
B_field = B_field.reshape((3,nx,ny))

im = ax[0].pcolormesh(r[0].reshape((nx,ny))/wavelength,r[1].reshape((nx,ny))/wavelength,E_field[2]/1e9,cmap=mymap,vmin=-1,vmax=1)
cb = fig.colorbar(im, ax=ax[0], orientation='horizontal',location='top', pad=0)   
cb.ax.xaxis.set_tick_params('both',direction='in')
cb.set_label(r'$E_x$ ($10^{9}$StatV/m)')

ax[0].set_xlabel(r'$x$ ($\lambda$)')
ax[0].set_ylabel(r'$y$ ($\lambda$)')

fall = 10
boundary_edge = (YMin,YMin - boundarySize)
r_rel = (boundary_edge[0]-r[1])/boundarySize
r_rel[r_rel<0] = 0
boundary = mask(r_rel,fall,timeStep)
boundary = 1-boundary.reshape((nx,ny))

im = ax[0].pcolormesh(r[0].reshape((nx,ny))/wavelength,r[1].reshape((nx,ny))/wavelength,boundary,cmap=mymap_red,vmin=0,vmax=1.02)
cb = plt.colorbar(im, ax=ax[1], pad=0)
cb.set_label('Masking function')
cb.ax.yaxis.set_tick_params('both',direction='in')
cb.ax.sharey(ax[1])
#cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0f}'.format(x)))
ax[1].tick_params(axis='y', direction='in',labelleft=False)

ax[0].hlines(boundary_edge[0]/wavelength, XMin/wavelength/2, XMax*1e4*2, 'k',linestyle='--',lw=0.5)
ax[0].text(2, boundary_edge[0]/wavelength+2, r'Boundary', ha='right', va='center')

ax[0].annotate('', xytext=(0, 0), xy=(7,-7),arrowprops=dict(facecolor='black',
                                                            width=0.05,
                                                            headwidth=3.,
                                                            headlength=3.,
                                                            lw=0.5))
ax[0].set_xlim(XMin/wavelength/2, XMax*1e4*2)
ax[0].set_ylim((YMin-boundarySize)/wavelength, YMax/wavelength/2)

for i,a in enumerate(ax):
    a.tick_params(which='both',direction='in')

str_ = ['a','b','c','d']
for i,a in enumerate(ax):
    a.text(0.03,0.90,'({})'.format(str_[i]),transform=a.transAxes)

fig.savefig('boundary_absorption.png',transparent=True,dpi=900)
fig

In [ ]:
pi = np.pi
electronMass = 9.1095e-28
electronCharge = 4.8032e-10 

wavelength = 1e-4 #units? cm?
nx, XMin, XMax = 2**8, -16*wavelength, 16*wavelength
ny, YMin, YMax = 2**8, -16*wavelength, 16*wavelength
dx, dy = (XMax - XMin)/nx, (YMax - YMin)/ny
timeStep = 0.2*dx/c

amplitude_a0 = 0.001
omega = 2*pi*c/wavelength
a0 = electronMass*c*omega/(-electronCharge)
fieldAmplitude = amplitude_a0*a0
PulseDuration = 5e-15
k = 2*pi/wavelength

boundarySize = 4*wavelength
rot = -np.pi/4
#supRate = 1 - exp(log(0.01)/(wavelength/(lightVelocity*timeStep)))

def setField(r, E, B):
    
    r_rot = (r[0]*np.cos(rot)+r[1]*np.sin(rot), -r[0]*np.sin(rot)+r[1]*np.cos(rot))

    # P-polarized
    Ep_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*c)**2)*np.cos(k*r_rot[0])
    Bp_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*c)**2)*np.cos(k*r_rot[0])

    # S-polarized
    Es_rot = fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*c)**2)*np.cos(k*r_rot[0])
    Bs_rot = -fieldAmplitude*np.exp(-(r_rot[0]**2+r_rot[1]**2)/(PulseDuration*c)**2)*np.cos(k*r_rot[0])


    E[0] = Ep_rot*np.sin(-rot)
    E[1] = Ep_rot*np.cos(rot) 
    B[2] = Bp_rot

    B[0] = Bs_rot*np.sin(-rot)
    B[1] = Bs_rot*np.cos(rot) 
    E[2] = Es_rot


    if r[1] < YMin + boundarySize: # basic absorber Ymin
        rate = 0
        E[1] = rate*E[1]
        E[2] = rate*E[2]
        B[1] = rate*B[1]
        B[2] = rate*B[2]

    elif r[1] > YMax - boundarySize: # basic absorber Ymax
        rate = 0
        E[1] = rate*E[1]
        E[2] = rate*E[2]
        B[1] = rate*B[1]
        B[2] = rate*B[2]
        
        
E = np.zeros((3,nx,ny))
B = np.zeros((3,nx,ny))
x = np.linspace(XMin,XMax,nx)
y = np.linspace(YMin,YMax,ny)
r = np.array(np.meshgrid(x,y))


for i in range(nx):
    for j in range(ny):
        setField(r[:,i,j],E[:,i,j],B[:,i,j])
